Acknowledgement: Consulted with Claude for the framework of code. AI-generated code in this project is manually debugged and heavily modified. As a result, the majority of the code in this file is manually written.

In [ ]:
import sqlite3
import csv

def create_flights_table(conn):
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_id TEXT PRIMARY KEY,
        source TEXT,
        destination TEXT,
        capacity INTEGER,
        seats_remaining INTEGER,
        dep_date TEXT,
        dep_hour TEXT,
        distance_km REAL,
        base_fare REAL
    );
    """
    # Modified: retain generated departure details, distance, and base fare.
    conn.execute(create_table_sql)
    # Added: persist orders for booking and cancellation.
    conn.execute("""CREATE TABLE IF NOT EXISTS bookings (
        booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
        flight_id TEXT NOT NULL REFERENCES flights(flight_id),
        tickets INTEGER NOT NULL CHECK(tickets > 0),
        status TEXT NOT NULL CHECK(status IN ('confirmed', 'cancelled')),
        booked_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
    )""")
    conn.commit()

def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    # Added: avoid re-importing the initial CSV over existing flight inventory.
    if cursor.execute("SELECT COUNT(*) FROM flights").fetchone()[0] > 0:
        print("Flights already loaded; existing inventory preserved.")
        return
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (row["flight_id"], row["source"], row["destination"], row["capacity"], row["seats_remaining"], row["dep_date"], row["dep_hour"], row["distance_km"], row["base_fare"])
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_id, source, destination, capacity, seats_remaining, dep_date, dep_hour, distance_km, base_fare)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name} table.")

if __name__ == "__main__":
    conn = sqlite3.connect("flights_database.db") # NOTE: Claude suggested saving to the database file (.db), so the data persists after the notebook session ends
    create_flights_table(conn)          # create the table first
    load_csv_to_table(conn, "flights.csv")
    conn.close()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("flights_database.db")  # Fixed: query the database created above.

def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

# 1. View all flights
print(query("SELECT * FROM flights"))

# 2. Flights with available seats
print(query("""
    SELECT flight_id, source, destination, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))

# 3. Fully booked flights
print(query("""
    SELECT flight_id, source, destination
    FROM flights
    WHERE seats_remaining = 0
"""))
